In [0]:
# Delta Time Travel
# See every change ever made to our bronze_orders table

print("=== HISTORY OF BRONZE_ORDERS TABLE ===\n")

history = spark.sql("DESCRIBE HISTORY bronze_orders")

display(history)

=== HISTORY OF BRONZE_ORDERS TABLE ===



version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-06-22T08:49:09.000Z,79042367658343,sritejareddy18@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(797120710706892),914e9e38-e50a-487d-aae4-e9bf8eeee79e,0622-084821-hukkejnq-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 99441, numOutputBytes -> 5718181)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13


In [0]:
# Query bronze_orders exactly as it was at Version 0
# This is the first time we ever loaded the data

print("=== QUERYING TABLE AT VERSION 0 ===\n")

version_0 = spark.sql("""
    SELECT * FROM bronze_orders VERSION AS OF 0
""")

print("Row count at version 0:", version_0.count())
display(version_0.limit(5))

=== QUERYING TABLE AT VERSION 0 ===

Row count at version 0: 99441


order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02T10:56:33.000Z,2017-10-02T11:07:15.000Z,2017-10-04T19:55:00.000Z,2017-10-10T21:25:13.000Z,2017-10-18T00:00:00.000Z
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24T20:41:37.000Z,2018-07-26T03:24:27.000Z,2018-07-26T14:31:00.000Z,2018-08-07T15:27:45.000Z,2018-08-13T00:00:00.000Z
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08T08:38:49.000Z,2018-08-08T08:55:23.000Z,2018-08-08T13:50:00.000Z,2018-08-17T18:06:29.000Z,2018-09-04T00:00:00.000Z
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18T19:28:06.000Z,2017-11-18T19:45:59.000Z,2017-11-22T13:39:59.000Z,2017-12-02T00:28:42.000Z,2017-12-15T00:00:00.000Z
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13T21:18:39.000Z,2018-02-13T22:20:29.000Z,2018-02-14T19:46:34.000Z,2018-02-16T18:17:02.000Z,2018-02-26T00:00:00.000Z


In [0]:
# Query table by timestamp
# Show table history with timestamps

print("=== TABLE HISTORY WITH TIMESTAMPS ===\n")

# Show history with key details only
spark.sql("""
    SELECT 
        version,
        timestamp,
        operation,
        operationMetrics.numOutputRows as rows_written
    FROM (DESCRIBE HISTORY bronze_orders)
""").show(truncate=False)

print("\n=== RESTORE EXAMPLE ===")
print("If data gets corrupted, you can restore like this:")
print("spark.sql('RESTORE TABLE bronze_orders TO VERSION AS OF 0')")
print("This brings back the table to version 0 instantly!")

=== TABLE HISTORY WITH TIMESTAMPS ===

+-------+-------------------+---------------------------------+------------+
|version|timestamp          |operation                        |rows_written|
+-------+-------------------+---------------------------------+------------+
|0      |2026-06-22 08:49:09|CREATE OR REPLACE TABLE AS SELECT|99441       |
+-------+-------------------+---------------------------------+------------+


=== RESTORE EXAMPLE ===
If data gets corrupted, you can restore like this:
spark.sql('RESTORE TABLE bronze_orders TO VERSION AS OF 0')
This brings back the table to version 0 instantly!


In [0]:
# Compare row counts across all versions
# Useful for spotting unexpected data changes

print("=== VERSION COMPARISON ===\n")

history = spark.sql("DESCRIBE HISTORY bronze_orders")

for row in history.collect():
    version = row['version']
    timestamp = row['timestamp']
    operation = row['operation']
    print(f"Version {version} | {timestamp} | Operation: {operation}")

print("\nDelta Time Travel setup complete!")
print("Your data is protected — every version is saved automatically!")

=== VERSION COMPARISON ===

Version 0 | 2026-06-22 08:49:09 | Operation: CREATE OR REPLACE TABLE AS SELECT

Delta Time Travel setup complete!
Your data is protected — every version is saved automatically!
